# UPDATE

V10:Data preprocessing has been added, resolving the issue of loss convergence.

V12:Added L2 regularization and filtering

In [1]:
import os,gc
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import glob
import torch
import warnings

# 禁用所有警告
warnings.filterwarnings('ignore')

2026-04-28 03:13:29.702079: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777346009.715072      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777346009.719131      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-28 03:13:29.734761: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

# Load Train data

In [2]:
PATH = '/kaggle/input/hms-harmful-brain-activity-classification/'
df = pd.read_csv(PATH + 'train.csv')
TARGETS = df.columns[-6:]
print('Train shape:', df.shape )
print('Targets', list(TARGETS))
df.head()

Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


,eeg_id,eeg_sub_id,eeg_label_offset_seconds,spectrogram_id,spectrogram_sub_id,spectrogram_label_offset_seconds,label_id,patient_id,expert_consensus,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,1628180742,0,0.0,353733,0,0.0,127492639,42516,Seizure,3,0,0,0,0,0
1,1628180742,1,6.0,353733,1,6.0,3887563113,42516,Seizure,3,0,0,0,0,0
2,1628180742,2,8.0,353733,2,8.0,1142670488,42516,Seizure,3,0,0,0,0,0
3,1628180742,3,18.0,353733,3,18.0,2718991173,42516,Seizure,3,0,0,0,0,0
4,1628180742,4,24.0,353733,4,24.0,3080632009,42516,Seizure,3,0,0,0,0,0


# Create Non-Overlapping Eeg Id Train Data

In [3]:
train = df.groupby('eeg_id')[['spectrogram_id','spectrogram_label_offset_seconds']].agg(
    {'spectrogram_id':'first','spectrogram_label_offset_seconds':'min'})
train.columns = ['spectrogram_id','min']

tmp = df.groupby('eeg_id')[['spectrogram_id','spectrogram_label_offset_seconds']].agg(
    {'spectrogram_label_offset_seconds':'max'})
train['max'] = tmp

tmp = df.groupby('eeg_id')[['patient_id']].agg('first')
train['patient_id'] = tmp

tmp = df.groupby('eeg_id')[TARGETS].agg('sum')
for t in TARGETS:
    train[t] = tmp[t].values
    
y_data = train[TARGETS].values
y_data = y_data / y_data.sum(axis=1,keepdims=True)
train[TARGETS] = y_data

tmp = df.groupby('eeg_id')[['expert_consensus']].agg('first')
train['target'] = tmp

train = train.reset_index()
print('Train non-overlapp eeg_id shape:', train.shape )

train.head(20)

Train non-overlapp eeg_id shape: (15396, 12)


,eeg_id,spectrogram_id,min,max,patient_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,target
0,568657,789577333,0.0,16.0,20654,0.000000,0.000000,0.250000,0.000000,0.166667,0.583333,Other
1,582999,1552638400,0.0,38.0,20230,0.000000,0.857143,0.000000,0.071429,0.000000,0.071429,LPD
2,642382,14960202,1008.0,1032.0,5955,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
3,1629671,2036345030,0.0,160.0,37481,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Seizure
4,1895581,128369999,1138.0,1138.0,47999,0.076923,0.000000,0.000000,0.000000,0.076923,0.846154,Other
5,2061593,320962633,1450.0,1450.0,23828,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
6,2078097,2074135650,3342.0,3342.0,61174,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
7,2366870,1232582129,0.0,30.0,23633,0.000000,0.333333,0.000000,0.000000,0.000000,0.666667,Other
8,2482631,978166025,1902.0,1944.0,20606,0.000000,0.000000,0.133333,0.066667,0.133333,0.666667,Other
9,2521897,673742515,0.0,4.0,62117,0.000000,0.000000,0.083333,0.083333,0.333333,0.500000,Other


In [4]:
import numpy as np

ycol = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
cd = {'Seizure': 'seizure_vote', 'GPD': 'gpd_vote', 'LRDA': 'lrda_vote', 'Other': 'other_vote', 'GRDA': 'grda_vote', 'LPD': 'lpd_vote'}

# 提取概率列和标签列
eeg_id_col = train.iloc[:, 0]  # 第一列是 eeg_id 列
prob_cols = train.iloc[:, -7:-1]  # 倒数第七到倒数第二列是概率列
label_col = train.iloc[:, -1]  # 最后一列是标签列

# 将概率列转换为 float32 类型
prob_cols = prob_cols.astype("float32")

# 归一化概率列
prob_cols_normalized = prob_cols.div(prob_cols.sum(axis=1), axis=0)

# 重新组合成 DataFrame
normalized_data = pd.concat([eeg_id_col, prob_cols_normalized, label_col], axis=1)

normalized_data.head(20)

,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,target
0,568657,0.000000,0.000000,0.250000,0.000000,0.166667,0.583333,Other
1,582999,0.000000,0.857143,0.000000,0.071429,0.000000,0.071429,LPD
2,642382,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
3,1629671,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,Seizure
4,1895581,0.076923,0.000000,0.000000,0.000000,0.076923,0.846154,Other
5,2061593,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
6,2078097,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,Other
7,2366870,0.000000,0.333333,0.000000,0.000000,0.000000,0.666667,Other
8,2482631,0.000000,0.000000,0.133333,0.066667,0.133333,0.666667,Other
9,2521897,0.000000,0.000000,0.083333,0.083333,0.333333,0.500000,Other


In [5]:
normalized_data.to_csv("normalized_data.csv", index=False)

In [6]:
EEG_PATH = '/kaggle/input/hms-harmful-brain-activity-classification/train_eegs/'
train_path = '/kaggle/input/processed/normalized_data.csv'

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import glob
from scipy.signal import butter, sosfilt
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

import pandas as pd

class EEGDataset(Dataset):
    def __init__(self, csv_file, eeg_path):
        self.csv = pd.read_csv(csv_file)  # 加载 CSV 文件
        self.eeg_path = eeg_path
        self.sos = self.butter_bandpass_filter_init()
        self.FEATS = ['Fp1','T3','C3','O1','Fp2','C4','T4','O2']  # 指定需要使用的特征列

    def butter_bandpass_filter_init(self):
        lowcut = 0.5  # 设置带通滤波的低频截止频率
        highcut = 40.0  # 设置带通滤波的高频截止频率
        fs = 200.0  # 采样频率
        order = 5  # 滤波器阶数

        nyq = 0.5 * fs
        low = lowcut / nyq
        high = highcut / nyq
        sos = butter(order, [low, high], analog=False, btype='band', output='sos')
        return sos

    def butter_bandpass_filter(self, data):
        y = sosfilt(self.sos, data)
        return y
    
    def __len__(self):
        return len(self.csv)
    
    def __getitem__(self, idx):
        eeg_id = self.csv.loc[idx, 'eeg_id']
        eeg_file_path = f"{self.eeg_path}{eeg_id}.parquet"  # 构建 EEG 数据文件的路径
        eeg_data = pd.read_parquet(eeg_file_path)[self.FEATS].values  # 从 Parquet 文件中加载指定的特征列
        # 检查是否存在 NaN 值
        if np.isnan(eeg_data).any():
            # 如果存在 NaN 值，你可以选择将其填充为特定的值或者进行插值处理
            # 这里我们使用 SimpleImputer 进行简单的填充处理，将 NaN 值替换为均值
            imputer = SimpleImputer(strategy='mean')
            eeg_data = imputer.fit_transform(eeg_data)
        eeg_data = self.butter_bandpass_filter(eeg_data)  # 对数据进行滤波
        eeg_data = torch.tensor(eeg_data, dtype=torch.float32)  # 转换为 PyTorch 张量
                    
        # 选择中间 10000 个时间点的数据
        mid_index = eeg_data.shape[0] // 2
        start_index = mid_index - 5000  # 从中间向左偏移5000个时间点
        end_index = mid_index + 5000  # 到中间向右偏移5000个时间点
        eeg_data = eeg_data[start_index:end_index]
        # 交换维度位置
        eeg_data = torch.transpose(eeg_data, 0, 1)
        
        labels = torch.tensor(self.csv.loc[idx, ycol].values.astype(np.float32), dtype=torch.float32)  # 加载对应的标签
        timestamps = np.arange(start_index, end_index)  # 创建时间戳数组，从start_index到end_index
        timestamps = torch.tensor(timestamps, dtype=torch.float32)  # 转换为PyTorch张量
                    
        return eeg_data, labels, timestamps

# 创建 EEG 数据集实例
dataset = EEGDataset(train_path, EEG_PATH)

# 创建数据加载器
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)


# 创建 EEG 数据集实例
dataset = EEGDataset(train_path, EEG_PATH)

# 创建数据加载器
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/processed/normalized_data.csv'

In [8]:
X, y, time = dataset[0]
print(f"Sample {0 + 1}: X shape {X.shape}, y shape {y.shape}, time shape {time.shape}")

NameError: name 'dataset' is not defined

# EEGnet Model

In [9]:
import torch.nn.functional as F

class TLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super(TLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, dropout=dropout, bidirectional=True)

    def forward(self, x, timestamps):
        output, (h_n, c_n) = self.lstm(x)
        return output, (h_n, c_n)


class EEGNet(nn.Module):

    def __init__(self, in_channels=8, num_classes=6):
        super(EEGNet, self).__init__()
        self.conv1 = nn.Conv1d(in_channels, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm1d(64)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)

        self.tlstm = TLSTM(input_size=64, hidden_size=128)

        self.attention = nn.Sequential(
            nn.Linear(128 * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.fc = nn.Linear(128 * 2, num_classes)

    def forward(self, x, timestamps):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        # reshape for TLSTM
        batch_size, channels, seq_length = x.size()
        x = x.permute(0, 2, 1)  # Change shape to [batch, seq_length, channels]

        # TLSTM layer
        x, _ = self.tlstm(x, timestamps)
        
        # Attention layer
        att_weights = F.softmax(self.attention(x), dim=1)
        x = torch.sum(att_weights * x, dim=1)

        # Fully connected layer
        x = self.fc(x)

        return x


In [10]:
# 定义一些超参数
input_channels = 8
num_classes = 6  # 类别数

# 创建EEGNet实例
model = EEGNet(in_channels=input_channels, num_classes=num_classes)

# 将模型放在GPU上
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs, labels, timestamps) in enumerate(dataloader):  # 添加timestamps参数
        inputs, labels = inputs.to(device), labels.to(device)
        timestamps = [len(seq) for seq in inputs]  # 计算每个序列的时间戳
        timestamps = torch.tensor(timestamps).to(device)
        optimizer.zero_grad()
        outputs = model(inputs, timestamps)  # 传递时间戳参数
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(dataloader)}")


NameError: name 'dataloader' is not defined

# Predict test data

In [11]:
test_path = '/kaggle/input/hms-harmful-brain-activity-classification/test.csv'
TEST_EEG_PATH = '/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/'

class TestEEGDataset(Dataset):
    def __init__(self, csv_file, eeg_path):
        self.csv = pd.read_csv(csv_file)  # 加载 CSV 文件
        self.eeg_path = eeg_path
        self.sos = self.butter_bandpass_filter_init()  # 初始化 Butterworth 滤波器参数
        self.FEATS = ['Fp1','T3','C3','O1','Fp2','C4','T4','O2']

    def __len__(self):
        return len(self.csv)

    def butter_bandpass_filter_init(self):
        lowcut = 0.5  # 设置带通滤波的低频截止频率
        highcut = 40.0  # 设置带通滤波的高频截止频率
        fs = 200.0  # 采样频率
        order = 5  # 滤波器阶数

        nyq = 0.5 * fs
        low = lowcut / nyq
        high = highcut / nyq
        sos = butter(order, [low, high], analog=False, btype='band', output='sos')  # 创建 Butterworth 滤波器的二阶段
        return sos

    def butter_bandpass_filter(self, data):
        y = sosfilt(self.sos, data)
        return y

    def __getitem__(self, idx):
        eeg_id = self.csv.loc[idx, 'eeg_id']
        eeg_file_path = f"{self.eeg_path}{eeg_id}.parquet"  # 构建 EEG 数据文件的路径
        eeg_data = pd.read_parquet(eeg_file_path)[self.FEATS].values  # 从 Parquet 文件中加载指定的特征列

        eeg_data = self.butter_bandpass_filter(eeg_data)  # 对数据进行滤波
        eeg_data = torch.tensor(eeg_data, dtype=torch.float32)  # 转换为 PyTorch 张量

        # 选择中间 10000 个时间点的数据
        mid_index = eeg_data.shape[0] // 2
        start_index = mid_index - 5000  # 从中间向左偏移5000个时间点
        end_index = mid_index + 5000  # 到中间向右偏移5000个时间点
        eeg_data = eeg_data[start_index:end_index]
        # 交换维度位置
        eeg_data = torch.transpose(eeg_data, 0, 1)

        return eeg_data


# 创建 EEG 数据集实例
testdataset = TestEEGDataset(test_path, TEST_EEG_PATH)

# 创建数据加载器
test_dataloader = DataLoader(testdataset, batch_size=32, shuffle=True, num_workers=4)

# 2. 将模型设置为评估模式
model.eval()

predictions = []
# 执行推理
with torch.no_grad():
    for inputs in test_dataloader:  # 注意这里不需要 labels
        inputs = inputs.to(device)
        # 在调用模型时传递一个 None 作为 timestamps 参数
        outputs = model(inputs, None)
        # 处理输出，得到类别概率值
        probabilities = torch.softmax(outputs, dim=1)
        predictions.append(probabilities.cpu().numpy())

# 4. 对预测结果进行处理，如转换为概率
predictions = np.concatenate(predictions, axis=0)

# 5. 输出预测结果
print(predictions)

[[nan nan nan nan nan nan]
 [nan nan nan nan nan nan]
 [nan nan nan nan nan nan]
 ...
 [nan nan nan nan nan nan]
 [nan nan nan nan nan nan]
 [nan nan nan nan nan nan]]


In [12]:
# 创建 DataFrame
columns = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
results_df = pd.DataFrame(predictions, columns=columns)

# 打印 DataFrame
print(results_df)

      seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  other_vote
0              NaN       NaN       NaN        NaN        NaN         NaN
1              NaN       NaN       NaN        NaN        NaN         NaN
2              NaN       NaN       NaN        NaN        NaN         NaN
3              NaN       NaN       NaN        NaN        NaN         NaN
4              NaN       NaN       NaN        NaN        NaN         NaN
...            ...       ...       ...        ...        ...         ...
9845           NaN       NaN       NaN        NaN        NaN         NaN
9846           NaN       NaN       NaN        NaN        NaN         NaN
9847           NaN       NaN       NaN        NaN        NaN         NaN
9848           NaN       NaN       NaN        NaN        NaN         NaN
9849           NaN       NaN       NaN        NaN        NaN         NaN

[9850 rows x 6 columns]


In [13]:
test_data = pd.read_csv(test_path)
sub = pd.DataFrame({'eeg_id':test_data.eeg_id.values})
sub[TARGETS] = results_df
sub.to_csv('submission.csv',index=False)
print('Submissionn shape',sub.shape)
sub.head()

Submissionn shape (9850, 7)


,eeg_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote
0,2578018731,NaN,NaN,NaN,NaN,NaN,NaN
1,2578018731,NaN,NaN,NaN,NaN,NaN,NaN
2,2578018731,NaN,NaN,NaN,NaN,NaN,NaN
3,2578018731,NaN,NaN,NaN,NaN,NaN,NaN
4,1706196508,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
sub.iloc[:,-6:].sum(axis=1)

0       0.0
1       0.0
2       0.0
3       0.0
4       0.0
       ... 
9845    0.0
9846    0.0
9847    0.0
9848    0.0
9849    0.0
Length: 9850, dtype: float32